# 05 — Evaluate All Models

Computes ROUGE-1/2/L on the test split for: TF-IDF baseline, TextRank baseline, fine-tuned BERTSUM, fine-tuned PEGASUS-PubMed.

In [ ]:
# Setup: clone repo + mount Drive + install deps.
REPO_URL = 'https://github.com/Captain-Uchiha/scientific-paper-summarizer.git'
!rm -rf /content/scientific-summarizer
!git clone -q {REPO_URL} /content/scientific-summarizer
import sys, os
sys.path.insert(0, '/content/scientific-summarizer')
os.chdir('/content/scientific-summarizer')

!pip -q install 'transformers>=4.40' datasets accelerate sentencepiece rouge-score evaluate sumy scikit-learn nltk tqdm keybert pymupdf
import nltk; nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)

PROJECT_DIR = '/content/drive/MyDrive/scientific-summarizer-data'

from google.colab import drivedrive.mount('/content/drive', force_remount=True)

In [ ]:
import json, torch
from tqdm import tqdm
from evaluation.rouge_eval import compute_rouge

PROC = f'{PROJECT_DIR}/dataset/processed'
records = [json.loads(l) for l in open(f'{PROC}/test.jsonl')]
articles  = [r['input_text']  for r in records]
abstracts = [r['target_text'] for r in records]
print('test size:', len(records))

results = {}

In [ ]:
# Baselines
from models import baselines as bl

tfidf_preds    = [bl.tfidf_summary(a, num_sentences=7) for a in tqdm(articles, desc='tfidf')]
textrank_preds = [bl.textrank_summary(a, num_sentences=7) for a in tqdm(articles, desc='textrank')]
results['TF-IDF']   = compute_rouge(tfidf_preds, abstracts)
results['TextRank'] = compute_rouge(textrank_preds, abstracts)
print(results)

In [ ]:
# BERTSUM (fine-tuned)
from transformers import AutoTokenizer
from models import bertsum as bs

tok = AutoTokenizer.from_pretrained('bert-base-uncased')
model = bs.BertSumExt('bert-base-uncased')
CKPT = f'{PROJECT_DIR}/models/bertsum/bertsum_epoch3.pt'
model.load_state_dict(torch.load(CKPT, map_location='cpu'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'; model.to(device).eval()

bertsum_preds = [bs.bertsum_summary(a, model, tok, num_sentences=7, device=device)
                 for a in tqdm(articles, desc='bertsum')]
results['BERTSUM'] = compute_rouge(bertsum_preds, abstracts)
print(results)

In [ ]:
# PEGASUS-PubMed (fine-tuned)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
FT_DIR = f'{PROJECT_DIR}/models/pegasus-pubmed-ft'
tok_p = AutoTokenizer.from_pretrained(FT_DIR)
model_p = AutoModelForSeq2SeqLM.from_pretrained(FT_DIR).to(device).eval()

@torch.no_grad()
def gen(text):
    inp = tok_p(text, max_length=1024, truncation=True, return_tensors='pt').to(device)
    out = model_p.generate(**inp, num_beams=4, min_length=80, max_length=256,
                           no_repeat_ngram_size=3, length_penalty=1.0, early_stopping=True)
    return tok_p.decode(out[0], skip_special_tokens=True)

pegasus_preds = [gen(a) for a in tqdm(articles, desc='pegasus')]
results['PEGASUS-PubMed-FT'] = compute_rouge(pegasus_preds, abstracts)
print(results)

In [ ]:
import pandas as pd
df = pd.DataFrame(results).T[['rouge1', 'rouge2', 'rougeL']]
df.columns = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
out_csv = f'{PROJECT_DIR}/outputs/results.csv'
import os; os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df.to_csv(out_csv)
print(df)
print('Saved', out_csv)